# My Query Has Been Running for 30 Minutes

Analysis queries are slow, the data you want to look at is not in the DW, and every team reports a different subscriber count — all three are really
**a problem of where the data sits**. In this notebook we build the four layers raw → staging →
dimension/fact → mart ourselves on synthetic data from a fictional subscription app, and see how the same question ("daily active subscribers and revenue")
gets answered differently from raw and from a mart. Instead of a production DW we use DuckDB, so the same structure is reproduced
inside the notebook with no separate server.

- **11.6.2. Pulling straight from raw** — join four CSVs in a single query to get the answer, and measure its time and length.
- **11.6.3. staging** — one view per source; clean up only names, types, duplicates and system events.
- **11.6.4. dimension / fact** — build a user daily-snapshot dimension and two facts: subscription events and daily active subscriptions.
- **11.6.5. mart** — fill 90 days with a batch that builds one partition at a time, and answer the same question in three lines.
- **11.6.6. Reloading a partition** — run the same date twice to confirm idempotency, and add a unique-key test.
- **11.6.7. Late-arriving events** — compare the numbers at each batch time and apply a 4-day reload window.

## 11.6.1. Preparing the practice data

This practice needs several tables — to reproduce the situation of a real service that has several sources. If you have no data yet, first run `python generate_data.py` in a terminal to create the five CSVs below in the `data/` folder. It covers 3,000 users over 90 days (2026-04-01 to 06-29), with a fixed seed.

| File | One row = | Main columns | Rows (approx.) |
| --- | --- | --- | --- |
| events.csv | one app event | event_id, user_id, event_name, session_id, event_time, received_time, properties(JSON) | 95K |
| subscription_events.csv | one subscription event | sub_event_id, user_id, event_type(purchase/renewal/cancel/refund), sku_id, is_trial, event_time, received_time, price_local, currency | 1,600 |
| user_snapshots.csv | user × date attributes | snapshot_date, user_id, platform, country, app_version, install_date | 136K |
| sku.csv | one product | sku_id, duration_days, trial_days, base_price_usd | 4 |
| exchange_rates.csv | currency × date exchange rate | date, currency, rate_to_usd | 450 |

The events contain re-sent duplicate rows and system events whose names start with `system_`, country codes come in mixed case, and some rows have a `received_time` up to 3 days later than their `event_time`. That is what raw data usually looks like in practice. In the subscription events, a trial start is recorded as a purchase with price 0, and a refund as a negative amount.

Run the two setup cells below first. The first cell installs the dependencies, and the next one creates the in-memory DuckDB connection reused throughout the notebook — the `con` that appears in every practice cell from here on is this connection.

In [1]:
# Install dependencies (skips quickly if already installed)
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
import time
import datetime as dt

import duckdb
import pandas as pd

con = duckdb.connect()   # in-memory connection — reused throughout the notebook

---
## 11.6.2. Pulling straight from raw — a scaled-down version of the 30-minute query

With no mart, using only the raw files, let's answer "active subscribers and that day's revenue (USD) by date, platform and country".
We define an active subscriber as one who stays active from a purchase or renewal for the SKU's duration (the trial period for a trial), and whose subscription ends on the day of a cancel or refund if there is one. For the user's platform and country we use that day's snapshot.
Deduplication, currency cleanup, expanding periods into days, the snapshot join and the exchange-rate join all go into a single query.

In [3]:
raw_sql = """
WITH sub AS (                     -- subscription events: dedupe + normalize names/currency + attach SKU duration
  SELECT s.user_id, LOWER(s.event_type) AS event_type, s.sku_id,
         CAST(s.event_time AS TIMESTAMP) AS event_time,
         s.price_local, UPPER(s.currency) AS currency,
         CAST(CASE WHEN s.is_trial THEN k.trial_days ELSE k.duration_days END AS INTEGER) AS period_days
  FROM read_csv_auto('data/subscription_events.csv') s
  JOIN read_csv_auto('data/sku.csv') k USING (sku_id)
  QUALIFY ROW_NUMBER() OVER (PARTITION BY s.sub_event_id ORDER BY s.received_time) = 1
),
periods AS (                      -- one purchase/renewal = one subscription period
  SELECT user_id, CAST(event_time AS DATE) AS start_date,
         CAST(event_time AS DATE) + period_days - 1 AS end_date
  FROM sub WHERE event_type IN ('purchase', 'renewal')
),
ends AS (                         -- cancel/refund ends the subscription that day
  SELECT user_id, MIN(CAST(event_time AS DATE)) AS end_date
  FROM sub WHERE event_type IN ('cancel', 'refund') GROUP BY user_id
),
expanded AS (                     -- expand each period into individual dates
  SELECT user_id, CAST(UNNEST(generate_series(start_date, end_date, INTERVAL 1 DAY)) AS DATE) AS date
  FROM periods
),
active AS (
  SELECT DISTINCT x.user_id, x.date
  FROM expanded x LEFT JOIN ends e USING (user_id)
  WHERE (e.end_date IS NULL OR x.date < e.end_date) AND x.date <= DATE '2026-06-29'
),
snap AS (                         -- user attributes as of that day
  SELECT snapshot_date, user_id, platform, UPPER(country) AS country
  FROM read_csv_auto('data/user_snapshots.csv')
),
revenue AS (                      -- that day's payments converted to USD (refunds are negative)
  SELECT CAST(s.event_time AS DATE) AS date, s.user_id,
         SUM(s.price_local * r.rate_to_usd) AS revenue_usd
  FROM sub s
  JOIN read_csv_auto('data/exchange_rates.csv') r
    ON r.date = CAST(s.event_time AS DATE) AND r.currency = s.currency
  GROUP BY 1, 2
)
SELECT a.date, sn.platform, sn.country,
       COUNT(DISTINCT a.user_id)        AS active_subscribers,
       COALESCE(SUM(rv.revenue_usd), 0) AS revenue_usd
FROM active a
JOIN snap sn ON sn.user_id = a.user_id AND sn.snapshot_date = a.date
LEFT JOIN revenue rv ON rv.user_id = a.user_id AND rv.date = a.date
GROUP BY 1, 2, 3
ORDER BY 1, 2, 3
"""

One query now holds deduplication, currency cleanup, period handling, the snapshot join and the exchange-rate join. There are probably two pieces of syntax you have not seen before. `QUALIFY` filters rows on the result of a window function, so you can apply a `ROW_NUMBER() = 1` condition directly without a subquery (besides DuckDB, BigQuery and Snowflake support it too). `generate_series` combined with `UNNEST` builds the list of dates from a start date to an end date and expands it into one row per date. Run it and measure the line count and time.

In [4]:
t0 = time.time()
raw_result = con.execute(raw_sql).df()
print(f"query: {len(raw_sql.strip().splitlines())} lines · {time.time() - t0:.2f}s · result: {len(raw_result):,} rows")
raw_result.tail()

query: 47 lines · 0.15s · result: 885 rows


,date,platform,country,active_subscribers,revenue_usd
880,2026-06-29,ios,BR,26,7.259845
881,2026-06-29,ios,DE,11,0.000000
882,2026-06-29,ios,JP,29,4.970250
883,2026-06-29,ios,KR,72,43.043700
884,2026-06-29,ios,US,32,0.000000


A 47-line query returns 885 rows. At notebook scale it finishes in under a second rather than 30 minutes, but what deserves attention is the shape of the query, not the time.
This query has to be rewritten every time the PO asks the question, and another analyst writes it slightly differently. The moment the colleague next to you says "revenue should be net of refunds" and
changes the query so that amounts on the cancel/refund day are also added into revenue, the two of you have different revenue numbers. Even when looking at the same source, without a proper mart you end up relying on different values.

---
## 11.6.3. staging — tidying each source, one at a time

Now we start stacking the layers. The five CSVs in `data/` correspond to the src (raw) layer of section 11.3.1. Following the rule that raw is left untouched as it arrived, we leave the files alone and stack layers on top.
Staging is one view per source file, and per the rules in section 11.3.2 it cleans up only names, types, duplicates and noise. **No business logic goes in.** First we clean up the two sources: events and subscription events.

In [5]:
con.execute("""
CREATE OR REPLACE VIEW stg_events AS
SELECT event_id, user_id, event_name, session_id,
       CAST(event_time AS TIMESTAMP)    AS event_time,
       CAST(received_time AS TIMESTAMP) AS received_time,
       CAST(event_time AS DATE)         AS event_date,
       CAST(received_time AS DATE)      AS received_date,
       properties
FROM read_csv_auto('data/events.csv')
WHERE user_id IS NOT NULL
  AND event_name NOT LIKE 'system%'                                            -- drop system events
  AND event_name <> 'heartbeat'
QUALIFY ROW_NUMBER() OVER (PARTITION BY event_id ORDER BY received_time) = 1;  -- drop re-sent duplicates

CREATE OR REPLACE VIEW stg_subscription_events AS
SELECT sub_event_id, user_id, LOWER(event_type) AS event_type, sku_id, is_trial,
       CAST(event_time AS TIMESTAMP)    AS event_time,
       CAST(received_time AS TIMESTAMP) AS received_time,
       CAST(event_time AS DATE)         AS event_date,
       CAST(received_time AS DATE)      AS received_date,
       price_local, UPPER(currency) AS currency
FROM read_csv_auto('data/subscription_events.csv')
QUALIFY ROW_NUMBER() OVER (PARTITION BY sub_event_id ORDER BY received_time) = 1;
""")
con.execute("SELECT COUNT(*) AS rows, COUNT(DISTINCT user_id) AS users FROM stg_subscription_events").df()

,rows,users
0,1597,1008


Clean up the remaining three sources (user snapshots, SKU, exchange rates) the same way, then compare row counts between raw and staging.

In [6]:
con.execute("""
CREATE OR REPLACE VIEW stg_user_snapshots AS
SELECT snapshot_date, user_id, LOWER(platform) AS platform, UPPER(country) AS country,
       app_version, install_date
FROM read_csv_auto('data/user_snapshots.csv');

CREATE OR REPLACE VIEW stg_sku AS
SELECT sku_id, duration_days, trial_days, base_price_usd FROM read_csv_auto('data/sku.csv');

CREATE OR REPLACE VIEW stg_exchange_rates AS
SELECT date, UPPER(currency) AS currency, rate_to_usd FROM read_csv_auto('data/exchange_rates.csv');
""")
con.execute("""
SELECT (SELECT COUNT(*) FROM read_csv_auto('data/events.csv')) AS raw_rows,
       (SELECT COUNT(*) FROM stg_events)                        AS stg_rows
""").df()

,raw_rows,stg_rows
0,95313,86040


Raw's 95,313 rows become 86,040 in staging, which shows that the duplicates and system events were removed. The query in 11.6.2 was doing this cleanup inside the query every single time.

---
## 11.6.4. dimension and fact — tables that describe and tables that measure

The dimensions are the user daily snapshot and the SKU; there are two facts. `fact_subscription_events` has one row per subscription event (a transaction fact), and
`fact_subscriber_daily` has one row per day a user is subscribed (a daily-snapshot fact). The second fact is what produces the semi-additive "active subscribers" count.

In [7]:
con.execute("""
-- dim: user × date snapshot — slowly changing attributes are simply captured every day
CREATE OR REPLACE TABLE dim_user_daily AS
SELECT snapshot_date AS date, user_id, platform, country, app_version, install_date,
       snapshot_date - install_date AS days_since_install
FROM stg_user_snapshots;

CREATE OR REPLACE VIEW dim_user_latest AS                 -- for when you need the 'current state'
SELECT * FROM dim_user_daily
QUALIFY ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY date DESC) = 1;

CREATE OR REPLACE TABLE dim_sku AS SELECT * FROM stg_sku;
""")
con.execute("SELECT COUNT(*) AS rows, COUNT(DISTINCT user_id) AS users, MAX(days_since_install) AS max_days FROM dim_user_daily").df()

,rows,users,max_days
0,136213,3000,89


The transaction fact is the staging subscription events with the exchange rate (USD conversion) and the SKU duration attached. Refunds are negative amounts, so they drop out of the totals naturally.

In [8]:
con.execute("""
-- fact 1: subscription events (grain: one row per purchase/renewal/cancel/refund event)
CREATE OR REPLACE TABLE fact_subscription_events AS
SELECT s.sub_event_id, s.event_date, s.received_date, s.user_id, s.sku_id, s.event_type, s.is_trial,
       s.event_time, s.price_local, s.currency,
       s.price_local * r.rate_to_usd AS price_usd,                            -- exchange-rate join
       CAST(CASE WHEN s.is_trial THEN k.trial_days ELSE k.duration_days END AS INTEGER) AS period_days
FROM stg_subscription_events s
LEFT JOIN stg_exchange_rates r ON r.date = s.event_date AND r.currency = s.currency
JOIN dim_sku k USING (sku_id);
""")
con.execute("""
SELECT event_type, COUNT(*) AS n, ROUND(SUM(price_usd), 2) AS usd
FROM fact_subscription_events GROUP BY 1 ORDER BY 1
""").df()

,event_type,n,usd
0,cancel,26,0.00
1,purchase,1405,6074.05
2,refund,13,-126.14
3,renewal,153,915.74


The snapshot fact is built by expanding each purchase/renewal period into individual dates and excluding everything after a cancel or refund.

In [9]:
con.execute("""
-- fact 2: daily active-subscription snapshot (grain: one row per subscribed user × date)
CREATE OR REPLACE TABLE fact_subscriber_daily AS
WITH periods AS (
  SELECT user_id, sku_id, is_trial, event_date AS start_date,
         event_date + period_days - 1 AS end_date
  FROM fact_subscription_events WHERE event_type IN ('purchase', 'renewal')
),
ends AS (
  SELECT user_id, MIN(event_date) AS end_date
  FROM fact_subscription_events WHERE event_type IN ('cancel', 'refund') GROUP BY user_id
),
expanded AS (
  SELECT user_id, sku_id, is_trial,
         CAST(UNNEST(generate_series(start_date, end_date, INTERVAL 1 DAY)) AS DATE) AS date
  FROM periods
)
SELECT x.date, x.user_id, x.sku_id, x.is_trial
FROM expanded x LEFT JOIN ends e USING (user_id)
WHERE (e.end_date IS NULL OR x.date < e.end_date) AND x.date <= DATE '2026-06-29'
QUALIFY ROW_NUMBER() OVER (PARTITION BY x.date, x.user_id ORDER BY x.is_trial) = 1;   -- unique key (date, user_id)
""")
con.execute("SELECT COUNT(*) AS rows, COUNT(DISTINCT user_id) AS users, MIN(date) AS d0, MAX(date) AS d1 FROM fact_subscriber_daily").df()

,rows,users,d0,d1
0,20955,1008,2026-04-01,2026-06-29


The unique key of `fact_subscription_events` is `sub_event_id`, and the unique key of `fact_subscriber_daily` is `(date, user_id)`. This is what a grain declaration looks like once it reaches the code.

One simplification: because we take the cancel/refund date as the first one per user (MIN), a user who re-subscribes after cancelling disappears from this fact. The practice data has no re-subscriptions, so the result is the same, but when you port this to real work you need to refine it so a cancel is matched only to the immediately preceding subscription period.


---
## 11.6.5. mart — the same question in three lines

We join the two facts with the user dimension on the same date and pre-aggregate at the `date × platform × country × SKU × trial flag (is_trial)` level. The trial/paid distinction declared as a dimension in section 11.4.2.3 is kept as-is; the subscription streak from 11.4.2.4 is left out for simplicity — try extending it yourself by adding a column to the fact.
The SQL that fills the mart is written to **target a single date (`{partition_date}`) only**. This is the rule "one batch touches one partition", a scaled-down version of the batch that runs every morning.
The `GROUP BY ALL` at the end of the query is syntax that automatically groups by every non-aggregate column in the SELECT (supported by DuckDB, BigQuery and Snowflake). The `{received_filter}` in the middle is a placeholder left as an empty string for now; we fill it in 11.6.7 to filter out late-arriving events.

In [10]:
MART_SQL = """
INSERT INTO mart_subscription_daily
WITH ev AS (                                  -- that day's subscription events
  SELECT f.event_date AS date, d.platform, d.country, f.sku_id, f.is_trial,
         f.event_type, f.price_usd, NULL AS active_user
  FROM fact_subscription_events f
  JOIN dim_user_daily d ON d.user_id = f.user_id AND d.date = f.event_date
  WHERE f.event_date = DATE '{partition_date}' {received_filter}
),
act AS (                                      -- that day's active subscribers
  SELECT a.date, d.platform, d.country, a.sku_id, a.is_trial,
         NULL AS event_type, 0 AS price_usd, a.user_id AS active_user
  FROM fact_subscriber_daily a
  JOIN dim_user_daily d ON d.user_id = a.user_id AND d.date = a.date
  WHERE a.date = DATE '{partition_date}'
)
SELECT date, platform, country, sku_id, is_trial,
       COUNT(CASE WHEN event_type = 'purchase' THEN 1 END) AS new_count,
       COUNT(CASE WHEN event_type = 'renewal'  THEN 1 END) AS renewal_count,
       COUNT(CASE WHEN event_type = 'cancel'   THEN 1 END) AS cancel_count,
       COUNT(CASE WHEN event_type = 'refund'   THEN 1 END) AS refund_count,
       SUM(price_usd)                                      AS revenue_usd,
       COUNT(DISTINCT active_user)                         AS active_subscribers
FROM (SELECT * FROM ev UNION ALL SELECT * FROM act)
GROUP BY ALL
"""

Create the mart table, then fill 90 days with the `run_partition` function, which deletes one partition and refills it. Notice how a 90-day backfill is nothing more than "run it 90 times with a different date".

In [11]:
con.execute("""
CREATE OR REPLACE TABLE mart_subscription_daily (
  date DATE, platform VARCHAR, country VARCHAR, sku_id VARCHAR, is_trial BOOLEAN,
  new_count BIGINT, renewal_count BIGINT, cancel_count BIGINT, refund_count BIGINT,
  revenue_usd DOUBLE, active_subscribers BIGINT
)""")

def run_partition(partition_date, received_filter=""):
    """One batch run = one partition. Delete it and refill it (insert-overwrite)."""
    con.execute("DELETE FROM mart_subscription_daily WHERE date = ?", [partition_date])
    con.execute(MART_SQL.format(partition_date=partition_date, received_filter=received_filter))

start, end = dt.date(2026, 4, 1), dt.date(2026, 6, 29)
for i in range((end - start).days + 1):          # 90-day backfill = 90 runs
    run_partition(start + dt.timedelta(days=i))
con.execute("SELECT COUNT(*) AS rows, MIN(date) AS d0, MAX(date) AS d1 FROM mart_subscription_daily").df()

,rows,d0,d1
0,5225,2026-04-01,2026-06-29


Now let's answer the same question as in 11.6.2, this time from the mart.

In [12]:
mart_sql = """
SELECT date, platform, country,
       SUM(active_subscribers) AS active_subscribers, SUM(revenue_usd) AS revenue_usd
FROM mart_subscription_daily GROUP BY ALL ORDER BY ALL
"""
t0 = time.time()
mart_result = con.execute(mart_sql).df()
print(f"query: {len(mart_sql.strip().splitlines())} lines · {time.time() - t0:.3f}s · result: {len(mart_result):,} rows")

cmp = raw_result.merge(mart_result, on=["date", "platform", "country"], suffixes=("_raw", "_mart"))
print("Active-subscriber difference, raw query vs mart (sum):", int((cmp.active_subscribers_raw - cmp.active_subscribers_mart).abs().sum()))
print("Revenue difference, raw query vs mart (sum, USD):", round((cmp.revenue_usd_raw - cmp.revenue_usd_mart).sum(), 2))
mart_result.tail()

query: 3 lines · 0.002s · result: 885 rows
Active-subscriber difference, raw query vs mart (sum): 0
Revenue difference, raw query vs mart (sum, USD): 126.14


,date,platform,country,active_subscribers,revenue_usd
880,2026-06-29,ios,BR,26.0,7.259845
881,2026-06-29,ios,DE,11.0,0.000000
882,2026-06-29,ios,JP,29.0,4.970250
883,2026-06-29,ios,KR,72.0,43.043700
884,2026-06-29,ios,US,32.0,0.000000


The same question is done in three lines of `SELECT … FROM mart … GROUP BY`, and the active subscriber counts are exactly identical to the raw query. Now whoever runs the query gets the same number.

But compare revenue (revenue_usd) the same way and this time the numbers differ. In this practice data the raw query comes out $126.14 higher. The raw query treats the cancel/refund day as inactive, so the negative refund amounts drop out in the revenue join as well, while the mart aggregates every subscription event and returns revenue with refunds deducted. This is exactly how the situation from the chapter's introduction — marketing and finance reporting different revenue — comes about. When definitions are scattered across queries, a difference like this is hard even to notice; when the definition is pinned down in the mart, the difference itself disappears.

One caveat has been left in this query on purpose. `SUM(active_subscribers)` is only correct when summing rows that share the same date (i.e. when collapsing platform, country and SKU). Summing across dates falls into the semi-additive trap from section 11.3.3. If you want weekly active subscribers, use the value on the last day of the week, or the average, not the sum.

---
## 11.6.6. Reloading a single partition — idempotency and the unique-key test

Suppose there was a problem with the June 14 data, and rebuild just that date. Because `run_partition` deletes and refills, the result is the same no matter how many times you run it. For comparison, we also run an INSERT once more without the DELETE.

In [13]:
target = dt.date(2026, 6, 14)
before = con.execute("SELECT COUNT(*), ROUND(SUM(revenue_usd), 2) FROM mart_subscription_daily WHERE date = ?", [target]).fetchone()
run_partition(target); run_partition(target)          # run twice
after  = con.execute("SELECT COUNT(*), ROUND(SUM(revenue_usd), 2) FROM mart_subscription_daily WHERE date = ?", [target]).fetchone()
print("before/after running twice:", before, after)       # must be identical

# For comparison: what if we only INSERT, without the DELETE?
con.execute(MART_SQL.format(partition_date=target, received_filter=""))
dup = con.execute("""
SELECT date, platform, country, sku_id, is_trial, COUNT(*) AS n
FROM mart_subscription_daily GROUP BY ALL HAVING COUNT(*) > 1
""").df()
print(f"unique-key violations: {len(dup)}")             # anything other than 0 is a test failure
run_partition(target)                                 # restore

before/after running twice: (70, 109.83) (70, 109.83)
unique-key violations: 70


Running twice leaves the row count and revenue unchanged, while INSERT-only catches 70 unique-key violations. This is the unique-key test described in sections 11.4.2.5 and 11.5.3. If the grain we declared, `(date, platform, country, sku_id, is_trial)`, has duplicates, a number greater than 0 comes out. In practice this query runs automatically after the batch, and anything other than 0 triggers an alert. Try adding not-null and accepted-values (`event_type IN (…)`) tests the same way, one line each.

---
## 11.6.7. Late-arriving events — the reload window

In the practice data, `received_time` is up to 3 days later than `event_time`. Some of the subscription events that happened on June 14 do not arrive until the 15th, 16th or 17th. If every morning we built only "yesterday's partition" from "the data that has arrived so far", how much of June 14 would be missing? We reproduce the state at each batch time to find out.

Note that active subscriber counts only reflect the delay if `fact_subscriber_daily` itself is rebuilt, so here we look only at the subscription-event side where `received_filter` applies — purchase/renewal counts and revenue.

In [14]:
def run_partition_as_of(partition_date, as_of):
    """Build the partition_date partition using only events that arrived by as_of (reproduces the batch-day view)."""
    run_partition(partition_date, received_filter=f"AND f.received_date <= DATE '{as_of}'")

rows = []
for lag in range(0, 4):                                # arrivals through D, D+1, D+2, D+3 = the D+1..D+4 morning batches
    run_partition_as_of(target, target + dt.timedelta(days=lag))
    n, usd = con.execute("SELECT SUM(new_count + renewal_count), ROUND(SUM(revenue_usd), 2) FROM mart_subscription_daily WHERE date = ?", [target]).fetchone()
    rows.append((f"D+{lag + 1} morning batch", int(n), usd))
pd.DataFrame(rows, columns=["batch_time", "purchases_renewals", "revenue_usd"])

,batch_time,purchases_renewals,revenue_usd
0,D+1 morning batch,13,82.31
1,D+2 morning batch,20,96.90
2,D+3 morning batch,24,104.84
3,D+4 morning batch,26,109.83


At the D+1 morning batch only 13 of 26 events (50%) are visible; it goes through 20 at D+2 and 24 at D+3, and only at the D+4 morning batch does it settle at 26. Revenue climbs along with it, from $82 to $110. That is why the daily batch rebuilds "the last 4 days" rather than "yesterday".

In [15]:
def daily_batch(run_date, window_days=4):
    """Daily morning batch: rebuild the last window_days partitions."""
    for lag in range(1, window_days + 1):
        run_partition(run_date - dt.timedelta(days=lag))

daily_batch(dt.date(2026, 6, 18))   # reloads the 6/14, 6/15, 6/16, 6/17 partitions

---
## Wrapping up

Try shrinking the reload window to 2 days (`window_days=2`) on the practice data and check how much goes missing.

In the notebook we created every view and table by hand, but in practice a transformation tool like dbt does this for you — it maps each file in the `sql/` folder to one table,
reads the dependencies between files to decide the execution order, and lets you add unique and not-null tests with a single line of YAML.
The `sql/` folder here is split into one file per layer so you can get a feel for that structure ahead of time.

If you want to port this to your own service's data, put your event and payment extracts in a separate path (do not overwrite the example CSVs), and adjust only the file paths and column names in the staging views — the same layer structure carries over as-is.